In [ ]:
%pip install torch wandb kaggle -Uq

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- training will run on CPU (still fast for this model).")

In [ ]:
import os
import getpass

import wandb

_wandb_api_key_env = os.environ.get("WANDB_API_KEY")
os.environ.pop("WANDB_API_KEY", None)
try:
    wandb.logout()
except Exception:
    pass

WANDB_ENTITY = "toberi23-free-university-of-tbilisi-"
WANDB_PROJECT = "Walmart-Recruiting-Store-Sales-Forecasting"

wandb_key = _wandb_api_key_env or getpass.getpass("Paste your W&B API key: ").strip()
wandb.login(key=wandb_key, relogin=True, force=True, verify=True)

who = wandb.Api().viewer
print("Logged in as:", who.username, "| entity:", who.entity)

In [ ]:
import time, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.base import BaseEstimator, RegressorMixin

import wandb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

WANDB_GROUP = "DLinear_Training"
BASE_TAGS = ["dlinear", "walmart-sales-forecasting"]
pd.set_option("display.width", 200)

In [ ]:
class MovingAverage(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, self.kernel_size - 1 - pad, 1)
        x_padded = torch.cat([front, x, end], dim=1)
        return self.avg(x_padded.permute(0, 2, 1)).permute(0, 2, 1)

class SeriesDecomposition(nn.Module):

    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAverage(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        return x - trend, trend

class DLinear(nn.Module):

    def __init__(self, lookback, horizon, n_channels, kernel_size=7, individual=True, normalize=True):
        super().__init__()
        assert kernel_size < lookback, "kernel_size must be smaller than the lookback window"
        self.decomp = SeriesDecomposition(kernel_size)
        self.individual = individual
        self.normalize = normalize
        self.n_channels = n_channels
        self.horizon = horizon

        if individual:
            self.seasonal_weight = nn.Parameter(torch.empty(n_channels, horizon, lookback))
            self.trend_weight = nn.Parameter(torch.empty(n_channels, horizon, lookback))
            self.seasonal_bias = nn.Parameter(torch.zeros(n_channels, horizon))
            self.trend_bias = nn.Parameter(torch.zeros(n_channels, horizon))
            for w in (self.seasonal_weight, self.trend_weight):
                nn.init.kaiming_uniform_(w, a=5 ** 0.5)
        else:
            self.seasonal_linear = nn.Linear(lookback, horizon)
            self.trend_linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        if self.normalize:
            mean = x.mean(dim=1, keepdim=True)
            std = x.std(dim=1, keepdim=True) + 1e-5
            x = (x - mean) / std

        seasonal, trend = self.decomp(x)
        seasonal = seasonal.permute(0, 2, 1)
        trend = trend.permute(0, 2, 1)

        if self.individual:
            seasonal_out = torch.einsum("bcl,chl->bch", seasonal, self.seasonal_weight) + self.seasonal_bias
            trend_out = torch.einsum("bcl,chl->bch", trend, self.trend_weight) + self.trend_bias
        else:
            seasonal_out = self.seasonal_linear(seasonal)
            trend_out = self.trend_linear(trend)

        out = (seasonal_out + trend_out).permute(0, 2, 1)
        if self.normalize:
            out = out * std + mean
        return out

In [ ]:
import sys
import subprocess
import zipfile

_LOCAL_DATA_DIR = os.path.join("data", "walmart-recruiting-store-sales-forecasting")
_KAGGLE_MOUNT_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"
_IN_COLAB = "google.colab" in sys.modules

def _download_via_kaggle_cli(target_dir):
    os.makedirs(target_dir, exist_ok=True)
    has_token = os.environ.get("KAGGLE_API_TOKEN") or os.path.exists(os.path.expanduser("~/.kaggle/access_token"))
    if not has_token:
        token = getpass.getpass(
            "Paste your Kaggle API token (kaggle.com -> profile picture -> Settings -> API -> "
            "Create New Token): "
        ).strip()
        os.environ["KAGGLE_API_TOKEN"] = token

    subprocess.run(
        ["kaggle", "competitions", "download", "-c", "walmart-recruiting-store-sales-forecasting",
         "-p", target_dir],
        check=True,
    )
    zip_path = os.path.join(target_dir, "walmart-recruiting-store-sales-forecasting.zip")
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target_dir)
        os.remove(zip_path)

COMP = os.environ.get("WALMART_DATA_DIR")
if not COMP:
    if os.path.isdir(_LOCAL_DATA_DIR):
        COMP = _LOCAL_DATA_DIR
    elif os.path.isdir(_KAGGLE_MOUNT_DIR):
        COMP = _KAGGLE_MOUNT_DIR
    elif _IN_COLAB:
        print("Running on Colab with no local data found -- downloading via the Kaggle API...")
        _download_via_kaggle_cli(_LOCAL_DATA_DIR)
        COMP = _LOCAL_DATA_DIR
    else:
        COMP = _KAGGLE_MOUNT_DIR
print("Reading competition data from:", COMP)

def load_merged(kind: str = "train") -> pd.DataFrame:
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    base = pd.read_csv(f"{COMP}/{kind}.csv.zip")
    base["Date"] = pd.to_datetime(base["Date"])
    stores = pd.read_csv(f"{COMP}/stores.csv")
    feats = pd.read_csv(f"{COMP}/features.csv.zip")
    feats["Date"] = pd.to_datetime(feats["Date"])
    feats = feats.drop(columns=["IsHoliday"])
    return (
        base.merge(stores, on="Store", how="left")
            .merge(feats, on=["Store", "Date"], how="left")
            .sort_values(["Store", "Dept", "Date"])
            .reset_index(drop=True)
    )

df_train = load_merged("train")
df_test = load_merged("test")
print("Train:", df_train.shape, "| Test:", df_test.shape)
df_train.head()

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday, dtype=bool), 5.0, 1.0)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))

assert abs(wmae([10, 20], [8, 15], [True, False]) - 15 / 6) < 1e-9

def wmae_breakdown(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    is_holiday = np.asarray(is_holiday, dtype=bool)
    holiday_mae = float(np.mean(np.abs(y_true[is_holiday] - y_pred[is_holiday]))) if is_holiday.any() else float("nan")
    nonholiday_mae = float(np.mean(np.abs(y_true[~is_holiday] - y_pred[~is_holiday]))) if (~is_holiday).any() else float("nan")
    return holiday_mae, nonholiday_mae

assert wmae_breakdown([10, 20], [8, 15], [True, False]) == (2.0, 5.0)

VAL_START = pd.Timestamp("2011-11-04")
VAL_END = pd.Timestamp("2012-07-27")

cv_train = df_train[df_train.Date < VAL_START].copy()
cv_val = df_train[(df_train.Date >= VAL_START) & (df_train.Date <= VAL_END)].copy()

print("CV train:", cv_train.shape, cv_train.Date.min().date(), "->", cv_train.Date.max().date())
print("CV val  :", cv_val.shape, cv_val.Date.min().date(), "->", cv_val.Date.max().date())
print("Val holiday weeks:", cv_val.loc[cv_val.IsHoliday, "Date"].dt.strftime("%Y-%m-%d").unique().tolist())

In [ ]:
def build_wide_matrix(df: pd.DataFrame) -> pd.DataFrame:
    key = df["Store"].astype(str) + "|" + df["Dept"].astype(str)
    wide = df.assign(_key=key).pivot_table(index="Date", columns="_key", values="Weekly_Sales")

    full_index = pd.date_range(wide.index.min(), wide.index.max(), freq="7D")
    wide = wide.reindex(full_index)

    filled = wide.ffill()
    filled = filled.fillna(filled.mean())
    filled = filled.fillna(0.0)
    return filled

_sample_wide = build_wide_matrix(cv_train)
print("Wide matrix shape (weeks, series):", _sample_wide.shape)
print("Remaining NaNs after fill:", int(_sample_wide.isna().sum().sum()))
del _sample_wide

In [ ]:
HORIZON_WEEKS = 39

def make_windows(wide: pd.DataFrame, lookback: int, horizon: int = HORIZON_WEEKS):
    arr = wide.to_numpy(dtype=np.float32)
    T = arr.shape[0]
    n_windows = T - lookback - horizon + 1
    if n_windows < 1:
        raise ValueError(
            f"Not enough history ({T} weeks) for lookback={lookback} + horizon={horizon}."
        )
    Xs = np.stack([arr[t:t + lookback] for t in range(n_windows)])
    Ys = np.stack([arr[t + lookback:t + lookback + horizon] for t in range(n_windows)])
    return Xs, Ys, n_windows

IS_HOLIDAY_BY_DATE = df_train.drop_duplicates("Date").set_index("Date")["IsHoliday"].sort_index()

print("Example window count, L=13:", make_windows(build_wide_matrix(cv_train), 13)[2])
print("Example window count, L=52:", make_windows(build_wide_matrix(cv_train), 52)[2])

In [ ]:
CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        self.dec_cutoffs_ = {year: anchor - pd.Timedelta(weeks=4) for year, anchor in self.christmas_anchors.items()}
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        pred_df["_row"] = np.arange(len(pred_df))

        adjusted = preds.copy()
        self.last_stats_ = {"n_bulge_departments": 0, "n_shifted_two_year": 0,
                             "n_shifted_one_year": 0, "total_abs_sales_shifted": 0.0}

        for year, anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            week_dates = {w: anchor - pd.Timedelta(weeks=52 - w) for w in [48, 49, 50, 51, 52]}
            week_lookup = {v: k for k, v in week_dates.items()}
            in_window = pred_df["Date"].isin(week_lookup)
            if not in_window.any():
                continue

            sub = pred_df.loc[in_window].copy()
            sub["week"] = sub["Date"].map(week_lookup)
            wide = sub.pivot_table(index=["Store", "Dept"], columns="week", values="Weekly_Sales")
            wide = wide.dropna(subset=[48, 49, 50, 51, 52])
            if wide.empty:
                continue

            avg_49_51 = wide[[49, 50, 51]].mean(axis=1)
            avg_48_52 = wide[[48, 52]].mean(axis=1)
            has_bulge = avg_49_51 >= self.bulge_threshold * avg_48_52

            train_start = self.train_start_.reindex(wide.index)
            has_two_years = train_start <= self.dec_cutoffs_.get(year - 1, pd.Timestamp.min)
            has_one_year = (train_start <= self.dec_cutoffs_.get(year, pd.Timestamp.min)) & (~has_two_years)

            shift_fraction = pd.Series(0.0, index=wide.index)
            shift_fraction[has_two_years] = 2.5 / 7
            shift_fraction[has_one_year] = 2.0 / 7
            shift_fraction[~has_bulge] = 0.0

            to_shift = shift_fraction[shift_fraction > 0].index
            if len(to_shift) == 0:
                continue

            shifted = wide.copy()
            for key in to_shift:
                vals = wide.loc[key, [48, 49, 50, 51, 52]].to_numpy(dtype=float)
                s = shift_fraction[key]
                shifted.loc[key] = [(1 - s) * vals[i] + s * vals[(i - 1) % 5] for i in range(5)]

            shifted_long = shifted.loc[to_shift].stack().rename("Weekly_Sales_shifted").reset_index()
            shifted_long["Date"] = shifted_long["week"].map(week_dates)
            merged = sub.merge(shifted_long[["Store", "Dept", "Date", "Weekly_Sales_shifted"]],
                                on=["Store", "Dept", "Date"], how="left")
            merged = merged.dropna(subset=["Weekly_Sales_shifted"])
            adjusted[merged["_row"].to_numpy()] = merged["Weekly_Sales_shifted"].to_numpy()

            delta_abs = (shifted.loc[to_shift] - wide.loc[to_shift]).abs().to_numpy().sum()
            self.last_stats_["n_bulge_departments"] += int(has_bulge.sum())
            self.last_stats_["n_shifted_two_year"] += int((shift_fraction == 2.5 / 7).sum())
            self.last_stats_["n_shifted_one_year"] += int((shift_fraction == 2.0 / 7).sum())
            self.last_stats_["total_abs_sales_shifted"] += float(delta_abs)

        return adjusted

DEFAULT_DLINEAR_PARAMS = dict(
    lookback_weeks=39, kernel_size=7, individual=True, normalize=True,
    learning_rate=0.01, weight_decay=0.0, epochs=60, batch_size=32,
    use_sample_weight=False, use_christmas_shift=False,
)

class DLinearForecastPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, lookback_weeks=39, kernel_size=7, individual=True, normalize=True,
                 learning_rate=0.01, weight_decay=0.0, epochs=60, batch_size=32,
                 horizon_weeks=HORIZON_WEEKS, use_sample_weight=False,
                 use_christmas_shift=False, christmas_bulge_threshold=1.10, device=None):
        self.lookback_weeks = lookback_weeks
        self.kernel_size = kernel_size
        self.individual = individual
        self.normalize = normalize
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.batch_size = batch_size
        self.horizon_weeks = horizon_weeks
        self.use_sample_weight = use_sample_weight
        self.use_christmas_shift = use_christmas_shift
        self.christmas_bulge_threshold = christmas_bulge_threshold
        self.device = device

    def _resolve_device(self):
        return self.device or ("cuda" if torch.cuda.is_available() else "cpu")

    def fit(self, X, y=None):
        device = self._resolve_device()
        wide = build_wide_matrix(X)
        self.columns_ = wide.columns
        self.col_mean_ = wide.mean()
        self.global_mean_ = float(wide.to_numpy().mean())

        Xs, Ys, n_windows = make_windows(wide, self.lookback_weeks, self.horizon_weeks)
        self.n_windows_ = n_windows

        Xs_t = torch.tensor(Xs, device=device)
        Ys_t = torch.tensor(Ys, device=device)

        n_channels = wide.shape[1]
        self.model_ = DLinear(
            self.lookback_weeks, self.horizon_weeks, n_channels,
            kernel_size=self.kernel_size, individual=self.individual, normalize=self.normalize,
        ).to(device)

        horizon_dates = pd.date_range(wide.index[self.lookback_weeks], periods=n_windows + self.horizon_weeks - 1, freq="7D")
        holiday_flags = IS_HOLIDAY_BY_DATE.reindex(horizon_dates).fillna(False).to_numpy()
        holiday_w_np = np.stack([np.where(holiday_flags[i:i + self.horizon_weeks], 5.0, 1.0) for i in range(n_windows)])
        weight_t = torch.tensor(holiday_w_np, dtype=torch.float32, device=device).unsqueeze(-1) if self.use_sample_weight else None

        opt = torch.optim.Adam(self.model_.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        self.model_.train()
        n = Xs_t.shape[0]
        for epoch in range(self.epochs):
            perm = torch.randperm(n, device=device)
            epoch_loss_sum = 0.0
            for i in range(0, n, self.batch_size):
                idx = perm[i:i + self.batch_size]
                opt.zero_grad()
                pred = self.model_(Xs_t[idx])
                if weight_t is not None:
                    loss = (weight_t[idx] * (pred - Ys_t[idx]).abs()).mean()
                else:
                    loss = (pred - Ys_t[idx]).abs().mean()
                loss.backward()
                opt.step()
                epoch_loss_sum += loss.item() * len(idx)
            if wandb.run is not None:
                wandb.log({"epoch": epoch, "epoch_loss": epoch_loss_sum / n})

        self.model_.eval()
        with torch.no_grad():
            train_pred = self.model_(Xs_t).cpu().numpy().astype(np.float64)
        train_true = Ys.astype(np.float64)
        diff = np.abs(train_true - train_pred)
        holiday_w_bc = holiday_w_np[:, :, None]
        self.train_wmae_ = float(np.sum(holiday_w_bc * diff) / (np.sum(holiday_w_np) * diff.shape[-1]))

        self.last_window_ = wide.to_numpy(dtype=np.float32)[-self.lookback_weeks:]
        self.forecast_start_ = wide.index[-1] + pd.Timedelta(weeks=1)

        if self.use_christmas_shift:
            self.christmas_adjuster_ = ChristmasWeekShiftAdjuster(
                bulge_threshold=self.christmas_bulge_threshold
            ).fit(X)
        return self

    def _forecast_grid(self):
        device = self._resolve_device()
        self.model_.eval()
        with torch.no_grad():
            x = torch.tensor(self.last_window_[None, :, :], device=device)
            pred = self.model_(x)[0].cpu().numpy()
        dates = pd.date_range(self.forecast_start_, periods=self.horizon_weeks, freq="7D")
        return pd.DataFrame(pred, index=dates, columns=self.columns_)

    def predict(self, X):
        grid = self._forecast_grid()
        long = grid.stack().rename("pred").reset_index()
        long.columns = ["Date", "_key", "pred"]

        lookup = X[["Store", "Dept", "Date"]].copy()
        lookup["_key"] = X["Store"].astype(str) + "|" + X["Dept"].astype(str)
        merged = lookup.merge(long, on=["_key", "Date"], how="left")
        preds = merged["pred"].to_numpy(copy=True)

        missing = np.isnan(preds)
        if missing.any():
            fallback = merged.loc[missing, "_key"].map(self.col_mean_)
            preds[missing] = fallback.fillna(self.global_mean_).to_numpy()

        if self.use_christmas_shift:
            preds = self.christmas_adjuster_.adjust(X, preds)
        return preds

def evaluate_config(config, train_full, val_full, verbose=False):
    pipeline = DLinearForecastPipeline(**config)
    pipeline.fit(train_full)
    preds = pipeline.predict(val_full)
    score = wmae(val_full["Weekly_Sales"], preds, val_full["IsHoliday"])
    holiday_mae, nonholiday_mae = wmae_breakdown(val_full["Weekly_Sales"], preds, val_full["IsHoliday"])
    if verbose:
        print(f"WMAE: {score:.2f}  (train WMAE: {pipeline.train_wmae_:.2f}, holiday MAE: {holiday_mae:.2f}, "
              f"non-holiday MAE: {nonholiday_mae:.2f}, training windows: {pipeline.n_windows_})")
    return score, pipeline.train_wmae_, holiday_mae, nonholiday_mae, pipeline.n_windows_

INNER_HORIZON = 7

INNER_HOLIDAY_ANCHORS = [pd.Timestamp("2011-02-11"), pd.Timestamp("2011-09-09")]
N_INNER_FOLDS = 3

def _fold_origins(wide_index, lookback, horizon=INNER_HORIZON, n_folds=N_INNER_FOLDS):
    T = len(wide_index)
    origins = []
    for anchor in INNER_HOLIDAY_ANCHORS:
        pos = int(wide_index.searchsorted(anchor))
        if pos >= lookback + horizon and pos + horizon <= T:
            origins.append(pos)
    n_recency = max(n_folds - len(origins), 1)
    origins += [T - horizon * k for k in range(n_recency, 0, -1)]
    origins = sorted(set(origins))
    if origins[0] < lookback + horizon:
        raise ValueError(
            f"Not enough history ({T} weeks) for lookback={lookback} with recency folds of "
            f"{horizon} weeks each (need >= {lookback + horizon} weeks before the earliest origin)."
        )
    return origins

def evaluate_config_rolling(config, train_full, n_folds=N_INNER_FOLDS, horizon=INNER_HORIZON, verbose=False):
    wide_full = build_wide_matrix(train_full)
    L = config["lookback_weeks"]
    origins = _fold_origins(wide_full.index, L, horizon, n_folds)

    train_cutoff_date = wide_full.index[origins[0]]
    truncated = train_full[train_full["Date"] < train_cutoff_date]

    pipeline = DLinearForecastPipeline(**{**config, "horizon_weeks": horizon})
    pipeline.fit(truncated)

    device = pipeline._resolve_device()
    arr = wide_full[pipeline.columns_].to_numpy(dtype=np.float32)

    fold_scores = []
    fold_rows_all = []
    pipeline.model_.eval()
    for origin in origins:
        window = arr[origin - L:origin]
        with torch.no_grad():
            x = torch.tensor(window[None, :, :], device=device)
            pred = pipeline.model_(x)[0].cpu().numpy()
        dates = wide_full.index[origin:origin + horizon]
        grid = pd.DataFrame(pred, index=dates, columns=pipeline.columns_)

        fold_rows = train_full[(train_full["Date"] >= dates[0]) & (train_full["Date"] <= dates[-1])]
        long = grid.stack().rename("pred").reset_index()
        long.columns = ["Date", "_key", "pred"]
        lookup = fold_rows[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]].copy()
        lookup["_key"] = fold_rows["Store"].astype(str) + "|" + fold_rows["Dept"].astype(str)
        merged = lookup.merge(long, on=["_key", "Date"], how="left")
        fallback = merged["_key"].map(pipeline.col_mean_)
        merged["pred"] = merged["pred"].fillna(fallback).fillna(pipeline.global_mean_)

        fold_score = wmae(merged["Weekly_Sales"], merged["pred"], merged["IsHoliday"])
        fold_scores.append(fold_score)
        fold_rows_all.append(merged)
        if verbose:
            print(f"    fold origin {dates[0].date()} -> {dates[-1].date()}: WMAE={fold_score:.2f}")

    pooled = pd.concat(fold_rows_all, ignore_index=True)
    holiday_mae, nonholiday_mae = wmae_breakdown(pooled["Weekly_Sales"], pooled["pred"], pooled["IsHoliday"])

    return {
        "wmae": float(np.mean(fold_scores)),
        "wmae_fold_std": float(np.std(fold_scores)),
        "train_wmae": pipeline.train_wmae_,
        "holiday_mae": holiday_mae,
        "nonholiday_mae": nonholiday_mae,
        "n_train_windows": pipeline.n_windows_,
        "fold_scores": fold_scores,
    }

def run_stage(stage_name, configs, base_config):
    results = []
    for cfg_overrides in configs:
        cfg = {**base_config, **cfg_overrides}
        tag = "_".join(f"{k}{v}" for k, v in cfg_overrides.items())
        run_name = f"DLinear_Training_Sweep_{stage_name}_{tag}"

        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name=run_name, job_type="training-sweep",
                          config={**cfg, "stage": stage_name}, tags=BASE_TAGS + ["training-sweep", f"stage-{stage_name}"],
                          reinit=True)
        t0 = time.time()
        m = evaluate_config_rolling(cfg, cv_train)
        elapsed = time.time() - t0
        wandb.log({"wmae": m["wmae"], "wmae_fold_std": m["wmae_fold_std"], "train_wmae": m["train_wmae"],
                    "holiday_mae": m["holiday_mae"], "nonholiday_mae": m["nonholiday_mae"],
                    "seconds": elapsed, "n_train_windows": int(m["n_train_windows"])})
        run.finish()

        results.append({**cfg_overrides, "wmae": m["wmae"], "wmae_fold_std": m["wmae_fold_std"],
                         "train_wmae": m["train_wmae"], "holiday_mae": m["holiday_mae"],
                         "nonholiday_mae": m["nonholiday_mae"], "seconds": elapsed,
                         "n_train_windows": m["n_train_windows"]})
        print(f"{run_name}: WMAE={m['wmae']:.2f}+/-{m['wmae_fold_std']:.2f} train_WMAE={m['train_wmae']:.2f} "
              f"holiday_MAE={m['holiday_mae']:.2f} nonholiday_MAE={m['nonholiday_mae']:.2f} "
              f"windows={m['n_train_windows']} ({elapsed:.1f}s)")

    return pd.DataFrame(results).sort_values("wmae").reset_index(drop=True)

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="DLinear_Training_Baseline", job_type="training",
                  config={**DEFAULT_DLINEAR_PARAMS, "fold": "B_calendar_aligned"}, tags=BASE_TAGS + ["training-baseline"])

t0 = time.time()
baseline_rolling = evaluate_config_rolling(DEFAULT_DLINEAR_PARAMS, cv_train, verbose=True)
baseline_wmae, baseline_train_wmae, baseline_holiday_mae, baseline_nonholiday_mae, baseline_windows = evaluate_config(
    DEFAULT_DLINEAR_PARAMS, cv_train, cv_val, verbose=True
)
elapsed = time.time() - t0

wandb.log({"wmae": baseline_wmae, "train_wmae": baseline_train_wmae, "holiday_mae": baseline_holiday_mae,
           "nonholiday_mae": baseline_nonholiday_mae, "seconds": elapsed, "n_windows": int(baseline_windows),
           "rolling_wmae": baseline_rolling["wmae"], "rolling_wmae_fold_std": baseline_rolling["wmae_fold_std"]})
run.finish()
print(f"Baseline WMAE (fold B, 39-week, honest): {baseline_wmae:.2f} (train WMAE: {baseline_train_wmae:.2f}, "
      f"holiday MAE: {baseline_holiday_mae:.2f}, non-holiday MAE: {baseline_nonholiday_mae:.2f}, "
      f"{elapsed:.1f}s, {baseline_windows} training windows)")
print(f"Baseline WMAE (walk-forward, {INNER_HORIZON}-week, what the sweep uses): "
      f"{baseline_rolling['wmae']:.2f} +/- {baseline_rolling['wmae_fold_std']:.2f}")

In [ ]:
joint_configs = [
    {"lookback_weeks": L, "kernel_size": K, "individual": I}
    for L in [13, 26, 39, 52]
    for K in [k for k in [3, 7, 13, 25, 51] if k < L]
    for I in [True, False]
]
print(f"Joint stage: {len(joint_configs)} configs (lookback_weeks x kernel_size x individual, jointly)")
joint_results = run_stage("lookback_kernel_individual", joint_configs, DEFAULT_DLINEAR_PARAMS)
joint_results

In [ ]:
best_lookback = int(joint_results.loc[0, "lookback_weeks"])
best_kernel = int(joint_results.loc[0, "kernel_size"])
best_individual = bool(joint_results.loc[0, "individual"])
config_after_joint = {**DEFAULT_DLINEAR_PARAMS, "lookback_weeks": best_lookback,
                       "kernel_size": best_kernel, "individual": best_individual}
print("Joint stage winner (lookback_weeks, kernel_size, individual):",
      (best_lookback, best_kernel, best_individual))
print("Config after joint stage:", config_after_joint)

In [ ]:
stageA_configs = [{"normalize": v} for v in [True, False]]
stageA_results = run_stage("normalize", stageA_configs, config_after_joint)
stageA_results

In [ ]:
best_normalize = bool(stageA_results.loc[0, "normalize"])
config_after_A = {**config_after_joint, "normalize": best_normalize}

stageB_configs = [{"learning_rate": v} for v in [0.001, 0.003, 0.005, 0.01, 0.02, 0.05]]
stageB_results = run_stage("learning_rate", stageB_configs, config_after_A)
stageB_results

In [ ]:
best_lr = float(stageB_results.loc[0, "learning_rate"])
config_after_B = {**config_after_A, "learning_rate": best_lr}

stageC_configs = [{"use_sample_weight": v} for v in [True, False]]
stageC_results = run_stage("use_sample_weight", stageC_configs, config_after_B)
stageC_results

In [ ]:
best_use_sw = bool(stageC_results.loc[0, "use_sample_weight"])
config_after_C = {**config_after_B, "use_sample_weight": best_use_sw}

stageD_configs = [{"weight_decay": v} for v in [0.0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]]
stageD_results = run_stage("weight_decay", stageD_configs, config_after_C)
stageD_results

In [ ]:
def describe_stage(name, results_df, param_cols, hypothesis):
    best_idx = 0
    worst_idx = len(results_df) - 1
    best_vals = {c: results_df.loc[best_idx, c] for c in param_cols}
    worst_vals = {c: results_df.loc[worst_idx, c] for c in param_cols}
    best_wmae = results_df.loc[best_idx, "wmae"]
    worst_wmae = results_df.loc[worst_idx, "wmae"]
    spread = worst_wmae - best_wmae
    print(f"[{name}] best={best_vals} (WMAE={best_wmae:.2f})  worst={worst_vals} (WMAE={worst_wmae:.2f})  spread={spread:.2f}")
    print(f"    Hypothesis: {hypothesis}")
    print()

describe_stage("lookback_kernel_individual (joint)", joint_results,
                ["lookback_weeks", "kernel_size", "individual"],
                "individual=False should win at long lookbacks (few training windows, per-channel "
                "weights overfit) while individual=True should win at short lookbacks (many windows, "
                "per-channel weights can exploit series-specific scale/shape) -- the winner should NOT "
                "be the same individual value across every lookback, unlike what the old staged search "
                "assumed by construction.")
describe_stage("normalize", stageA_results, ["normalize"],
                "matters most when individual=False")
describe_stage("learning_rate", stageB_results, ["learning_rate"],
                "too high overfits/diverges, too low underfits in 60 epochs")
describe_stage("use_sample_weight", stageC_results, ["use_sample_weight"],
                "holiday-weighted loss should help WMAE specifically, now that the walk-forward folds "
                "actually include holiday weeks to measure that on")
describe_stage("weight_decay", stageD_results, ["weight_decay"],
                "a nonzero value should still help individual=True configs; less critical for "
                "individual=False, which has far fewer parameters to overfit with")

In [ ]:
best_weight_decay = float(stageD_results.loc[0, "weight_decay"])
best_config = {**config_after_C, "weight_decay": best_weight_decay}
print("Final selected config:", best_config)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="DLinear_Training_Final", job_type="training-full",
                  config={**best_config, "fold": "B_calendar_aligned"}, tags=BASE_TAGS + ["training-final"])

t0 = time.time()
final_wmae, final_train_wmae, final_holiday_mae, final_nonholiday_mae, final_windows = evaluate_config(
    best_config, cv_train, cv_val, verbose=True
)
elapsed = time.time() - t0

wandb.log({"wmae": final_wmae, "train_wmae": final_train_wmae, "holiday_mae": final_holiday_mae,
           "nonholiday_mae": final_nonholiday_mae, "seconds": elapsed, "n_windows": int(final_windows)})
run.finish()
print(f"Final WMAE (fold B, full 39-week horizon): {final_wmae:.2f}  (train WMAE: {final_train_wmae:.2f}, "
      f"holiday MAE: {final_holiday_mae:.2f}, non-holiday MAE: {final_nonholiday_mae:.2f})")

In [ ]:
christmas_config = {**best_config, "use_christmas_shift": True}

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
                  name="DLinear_Training_Final_ChristmasShift", job_type="training-full",
                  config={**christmas_config, "fold": "B_calendar_aligned"},
                  tags=BASE_TAGS + ["training-final", "christmas-shift"])

t0 = time.time()
final_wmae_shifted, final_train_wmae_shifted, final_holiday_mae_shifted, final_nonholiday_mae_shifted, _ = evaluate_config(
    christmas_config, cv_train, cv_val, verbose=True
)
elapsed = time.time() - t0

wandb.log({"wmae": final_wmae_shifted, "train_wmae": final_train_wmae_shifted, "holiday_mae": final_holiday_mae_shifted,
           "nonholiday_mae": final_nonholiday_mae_shifted, "seconds": elapsed})
run.finish()
print(f"WMAE with Christmas shift: {final_wmae_shifted:.2f}  (without: {final_wmae:.2f})  "
      f"[holiday MAE {final_holiday_mae:.2f} -> {final_holiday_mae_shifted:.2f}]")

In [ ]:
deploy_config = {**best_config, "use_christmas_shift": bool(final_wmae_shifted < final_wmae)}

final_pipeline = DLinearForecastPipeline(**deploy_config)
final_pipeline.fit(df_train)

final_pipeline.model_.to("cpu")
final_pipeline.device = "cpu"

with open("dlinear_pipeline.pkl", "wb") as f:
    pickle.dump(final_pipeline, f)

artifact_metadata = {
    **deploy_config,
    "cv_wmae_full": final_wmae,
    "cv_train_wmae_full": final_train_wmae,
    "cv_holiday_mae_full": final_holiday_mae,
    "cv_nonholiday_mae_full": final_nonholiday_mae,
    "cv_wmae_christmas_shift": final_wmae_shifted,
    "deploy_train_wmae": final_pipeline.train_wmae_,
}

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="DLinear_Pipeline_Save", job_type="save-pipeline",
                  config=artifact_metadata, tags=BASE_TAGS + ["pipeline-export"])

artifact = wandb.Artifact("dlinear_forecast_pipeline", type="model", metadata=artifact_metadata)
artifact.add_file("dlinear_pipeline.pkl")
run.log_artifact(artifact)
wandb.log({"cv_wmae_full": final_wmae, "train_wmae": final_train_wmae, "holiday_mae": final_holiday_mae,
           "nonholiday_mae": final_nonholiday_mae, "deploy_train_wmae": final_pipeline.train_wmae_})
run.finish()

print("Saved dlinear_pipeline.pkl and logged it as a W&B Artifact.")